# 03 — Score baselines on the August smoke test

Train the historical baselines on all pre-August data and score them on exactly the untouched period used by v3.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / 'common').exists() else ROOT.parent
sys.path.insert(0, str(ROOT / 'modeling'))

import numpy as np
import pandas as pd

from baselines import climatology, mae, persistence, rmse, xgb_baseline
from splits import time_split
from windowing import build_supervised

df = pd.read_parquet(ROOT / 'modeling' / 'feat_airquality.parquet')
X, y, meta = build_supervised(df)
train, validation, test = time_split(
    meta,
    validation_start='2025-07-01',
    test_start='2026-08-01',
)
development = np.concatenate([train, validation])

predictions = {
    '24h_persistence': persistence(X[test]),
    'climatology': climatology(
        X[development], y[development], meta.iloc[development],
        X[test], meta.iloc[test],
    ),
    'xgboost': xgb_baseline(X[development], y[development], X[test]),
}

high = y[test] > 35.4
rows = []
for name, prediction in predictions.items():
    rows.append({
        'model': name,
        'MAE': round(mae(prediction, y[test]), 3),
        'RMSE': round(rmse(prediction, y[test]), 3),
        'high_MAE': round(mae(prediction[high], y[test][high]), 3),
        'exceedance_recall': round(float((prediction[high] > 35.4).mean()), 3),
        'maximum_prediction': round(float(prediction.max()), 3),
    })

table = pd.DataFrame(rows)
print(table.to_string(index=False))

output = ROOT / 'modeling' / 'artifacts' / 'v3'
output.mkdir(parents=True, exist_ok=True)
table.to_csv(output / 'baseline_metrics.csv', index=False)
print('saved:', output / 'baseline_metrics.csv')